# Exemplo didático do agente

Este notebook é um primeiro contato com o agente de apoio e demonstra o fluxo **prompt → modelo → tool → resposta**.

Este exemplo cria um pequeno grafo e não inicia FastAPI/Uvicorn. Ele reutiliza o ambiente Python, o `.env`, o prompt oficial e o catálogo do projeto da pasta `backend`.

As últimas células chamam o Foundry, enviam o prompt e a pergunta fictícia e podem
consumir créditos. O recorte é alimentação por zona: não demonstra a flag de urgência, a interface
ou toda a infraestrutura do agente da aplicação. Sem persistência ou GPS real.

## Instalar dependências

Num terminal na raiz do projeto:

```sh
cd backend
uv sync --locked --group notebook
```

Abra este notebook no VS Code com suporte a Jupyter e selecione como kernel
`backend/.venv/Scripts/python.exe`.

A pasta do notebook não precisa coincidir com a do ambiente. Preencha as variáveis Foundry em `backend/.env`, conforme `backend/.env.example`, e execute as células na ordem.

## Ler a configuração
O `.env` é lido explicitamente em `backend/`; variáveis do processo têm precedência. Não exibimos seus valores.

In [6]:
import os
import json
from pathlib import Path
from urllib.parse import urlparse

from dotenv import dotenv_values
from pydantic import SecretStr

ROOT = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / "backend/pyproject.toml").is_file()
        and (folder / "data/processed/services.sample.json").is_file()
    ),
    None,
)

if ROOT is None:
    raise RuntimeError("Abra o notebook a partir de uma pasta deste repositório.")

config = {**dotenv_values(ROOT / "backend/.env"), **os.environ}

required = [
    "CARECOMPASS_FOUNDRY_BASE_URL",
    "CARECOMPASS_FOUNDRY_DEPLOYMENT",
    "CARECOMPASS_FOUNDRY_API_KEY",
]

if any(not config.get(name) for name in required):
    raise RuntimeError(
        "Preencha BASE_URL, DEPLOYMENT e API_KEY do Foundry em backend/.env; consulte .env.example."
    )

base_url = config["CARECOMPASS_FOUNDRY_BASE_URL"]
parsed = urlparse(base_url)

if (
    parsed.scheme != "https"
    or not parsed.hostname
    or parsed.username
    or parsed.password
    or parsed.query
    or parsed.fragment
    or not parsed.path.rstrip("/").endswith("/openai/v1")
):
    raise RuntimeError("Use o endpoint HTTPS de inferência terminado em /openai/v1/.")

print("Configuração encontrada. Valores e secrets não são exibidos.")

Configuração encontrada. Valores e secrets não são exibidos.


## Carregar o prompt e escolher a zona
O prompt é o arquivo oficial. Ao editá-lo, execute esta célula e as seguintes novamente. Caminhos personalizados relativos são resolvidos a partir de `backend/`, como ao iniciar a API por essa pasta.

In [7]:
def configured_path(variable, default):
    value = config.get(variable)
    if not value:
        return default
    path = Path(value)
    return path if path.is_absolute() else ROOT / "backend" / path


PROMPT_PATH = configured_path(
    "CARECOMPASS_PROMPT_PATH", ROOT / "backend/prompts/assistant.md"
)

CATALOG_PATH = configured_path(
    "CARECOMPASS_CATALOG_PATH", ROOT / "data/processed/services.sample.json"
)

prompt = PROMPT_PATH.read_text(encoding="utf-8").strip()

if not prompt:
    raise RuntimeError("O prompt oficial está vazio.")

ZONE = "leste"  # Experimente: centro, norte, sul ou oeste.
if ZONE not in {"centro", "leste", "norte", "sul", "oeste"}:
    raise ValueError("Zona inválida.")

services = json.loads(CATALOG_PATH.read_text(encoding="utf-8"))["services"]
selected = [item for item in services if item["zone"] == ZONE]

print(f"Zona selecionada: {ZONE}. Equipamentos na amostra: {len(selected)}.")

Zona selecionada: leste. Equipamentos na amostra: 8.


## Criar uma tool simples
A função consulta o JSON sem IA e impede consultar outra zona. O rótulo, endereço e descrição vêm da mesma amostra usada pela aplicação.

In [8]:
from langchain_core.tools import tool


@tool
def list_services(zone: str) -> str:
    """Consulte os equipamentos da zona ativa antes de recomendar alimentação."""
    if zone != ZONE:
        return json.dumps({"error": "zone_mismatch", "active_zone": ZONE})
    return json.dumps(
        {
            "zone": ZONE,
            "services": [
                {
                    key: item[key]
                    for key in ("id", "name", "type", "address", "description")
                }
                for item in selected
            ],
        },
        ensure_ascii=False,
    )


# Consulta determinística: esta linha não chama o modelo.
assert len(json.loads(list_services.invoke({"zone": ZONE}))["services"]) == len(
    selected
)

print("Tool pronta para consultar a zona selecionada.")

Tool pronta para consultar a zona selecionada.


## Criar o cliente e montar o agente

O modelo precisa suportar tool calling. Construímos dois nós: o assistente decide consultar a tool; depois recebe os dados para responder. Não é necessário configurar `CARECOMPASS_CHAT_ENGINE`: o notebook usa diretamente o cliente.

In [9]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

timeout_seconds = float(config.get("CARECOMPASS_AGENT_TIMEOUT_SECONDS", "15"))
max_steps = int(config.get("CARECOMPASS_AGENT_MAX_STEPS", "6"))

model = ChatOpenAI(
    base_url=base_url,
    model=config["CARECOMPASS_FOUNDRY_DEPLOYMENT"],
    api_key=SecretStr(config["CARECOMPASS_FOUNDRY_API_KEY"]),
    timeout=timeout_seconds,
    max_completion_tokens=int(config.get("CARECOMPASS_AGENT_MAX_OUTPUT_TOKENS", "600")),
    max_retries=0,
    use_responses_api=False,
)

model_with_tools = model.bind_tools([list_services])

system_message = SystemMessage(
    content=(
        prompt
        + f"\n\nZona ativa da aplicação: {ZONE}. A seleção prevalece sobre o histórico."
        "\nExemplo didático restrito a alimentação. A única tool disponível é list_services."
        " Não há contatos de urgência disponíveis neste exemplo."
        " Consulte a tool e recomende uma ou duas opções da zona, com endereço."
        " Não afirme proximidade: só conhecemos a zona, não coordenadas."
    )
)


async def assistant(state: MessagesState):
    reply = await model_with_tools.ainvoke([system_message, *state["messages"]])
    return {"messages": [reply]}


builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode([list_services]))
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")
agent = builder.compile() # Sem checkpoint ou persistência de conversas.

print("Agente criado; nenhuma chamada ao Foundry foi feita até aqui.")

Agente criado; nenhuma chamada ao Foundry foi feita até aqui.


## Executar uma pergunta
Esta célula faz a chamada real. A pergunta é fixa e a zona aparece explicitamente no resultado. Cada execução usa apenas esta mensagem; não reaproveita resultados anteriores como histórico. A tool consultada é mostrada para inspeção.

In [10]:
import asyncio
from langsmith import tracing_context

TEST_MESSAGE = "Onde posso encontrar alimentação barata na zona selecionada?"
state: MessagesState = {"messages": [HumanMessage(content=TEST_MESSAGE)]}

with tracing_context(enabled=False):
    async with asyncio.timeout(timeout_seconds):
        result = await agent.ainvoke(state, {"recursion_limit": max_steps})

used_tools = [
    message.name for message in result["messages"] if isinstance(message, ToolMessage)
]

print(f"Zona utilizada: {ZONE}")
print(f"Pergunta: {TEST_MESSAGE}")
print(f"Tools consultadas: {', '.join(used_tools) if used_tools else 'nenhuma'}")
print("Resposta:")
print(result["messages"][-1].content)

Zona utilizada: leste
Pergunta: Onde posso encontrar alimentação barata na zona selecionada?
Tools consultadas: list_services
Resposta:
Opções de alimentação barata na zona leste:

- **Bom Prato Refeitório Cidade Tiradentes** - Rua Edson Danillo Dotto, 39 – Zona Leste. Restaurante de alimentação a baixo custo.
- **Rede Cozinha Cidadã — Avenida Souza Ramos, 513** - Avenida Souza Ramos, 513 - Cidade Tiradentes, CEP: 08490-490. Ponto de distribuição de refeições.


## O que observar

- A resposta indica somente opções da zona escolhida e endereços do catálogo?
- O modelo consultou `list_services` antes de recomendar?
- A resposta é curta e não promete o local "mais próximo", vaga ou horário atual?

Para trocar a zona, altere `ZONE` e execute novamente as células de **Carregar o prompt e escolher a zona** até **Executar uma pergunta**. Para informar novamente a configuração, reexecute a célula **Ler a configuração**. Testes técnicos com modelo simulado não comprovam qualidade das respostas reais. Este notebook não inicia o backend, não implementa o painel/alerta de urgência e não substitui seus testes.